# Triagegeist: Clinical Intensity Prediction in the Emergency Context

### **Authors:** [Amey Thakur](https://www.kaggle.com/ameythakur20) & [Archit Konde](https://www.kaggle.com/architkonde)

***

## **1. Executive Research Summary**

Emergency triage systems provide the initial, high-stakes decision logic for hospital resource allocation. Across clinical platforms, the **Emergency Severity Index (ESI)** is used to classify patient acuity from Level 1 (Immediate resuscitation) to Level 5 (Non-urgent). Errors in this classification can lead to delayed intervention or preventable adverse events.

This research implements a **Multi-Tier Clinical Decision Support (CDS) System** using synthetic triage data from the **Laitinen-Fredriksson Foundation**. The architecture prioritizes **ESI-1 and ESI-2 identification** through a combination of pattern-matching (Tier 1), diagnostic specialization (Tier 2), and a balanced gradient-boosted fallback (Tier 3). 

### **Core Methodology:**
1. **Deterministic Pattern Mapping:** Capturing unambiguous high-risk chief complaints.
2. **Diagnostic Specialization:** Sub-models for symptoms requiring specific physiological thresholds.
3. **Balanced Generalization:** LightGBM-based classification of complex cases using consolidated vitals and comorbidities.

**Citation:** Olaf Yunus Laitinen Imanov (2026). Triagegeist. https://kaggle.com/competitions/triagegeist, 2026. Kaggle.

## **2. Environment Governance**

Strict dependency control and seed locking ensure reproducibility across environments. We integrate the `kaggle_toolbox` to manage memory and hardware state.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import quadratic_weighted_kappa as qwk_func
from sklearn.metrics import classification_report, confusion_matrix

try:
    import kaggle_toolbox as tb
except ImportError:
    tb = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

class CFG:
    SEED = 42
    TARGET = 'triage_acuity'
    N_FOLDS = 5
    # Standard ESI mapping logic
    ESI_MAP = {1: 'ESI-1', 2: 'ESI-2', 3: 'ESI-3', 4: 'ESI-4', 5: 'ESI-5'}

def seed_everything(seed=42):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if tb:
        tb.seed_everything(seed)
    
seed_everything(CFG.SEED)
print("Hardware Governance: Seed Locked (deterministic state).")

## **3. Relational Data Synthesis**

The dataset consists of multiple relational tables linked via `patient_id`. We merge these to form a unified clinical profile for each record.

In [ ]:
DATA_DIR = Path('/kaggle/input/triagegeist')
if not DATA_DIR.exists():
    # Local discovery engine for development
    DATA_DIR = Path('C:/Users/AMEY THAKUR/Downloads')

def load_and_merge(is_train=True):
    prefix = 'train' if is_train else 'test'
    
    df = pd.read_csv(DATA_DIR / f'{prefix}.csv')
    complaints = pd.read_csv(DATA_DIR / 'chief_complaints.csv')
    history = pd.read_csv(DATA_DIR / 'patient_history.csv')
    
    # Memory Optimization via Toolbox
    if tb: df = tb.reduce_mem_usage(df)
    
    # Relational Join: patient_id linkage binds profiles
    df = df.merge(complaints, on='patient_id', how='left')
    df = df.merge(history, on='patient_id', how='left')
    
    return df

train_raw = load_and_merge(is_train=True)
test_raw = load_and_merge(is_train=False)

print(f"Database Synthesis Complete. Clinical cohort: {len(train_raw):,} records.")
train_raw.head(3)

## **4. Exploratory Clinical Data Analysis (ECDA)**

We visualize the target distribution and vitals volatility to identify clinical regimes of high variance. Understanding the clinical context of missingness is key to triage modelling.

In [ ]:
def plot_clinical_volatility(df, target_col):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Target distribution: Assessing triage class balance
    sns.countplot(x=target_col, data=df, ax=axes[0], palette='viridis')
    axes[0].set_title('ESI Level Distribution: Triage Frequency')
    
    # Vitals vs Acuity: Assessing signal strength in NEWS2
    if 'news2_score' in df.columns:
        sns.boxplot(x=target_col, y='news2_score', data=df, ax=axes[1], palette='magma')
        axes[1].set_title('NEWS2 Correlation with Triage Acuity')
    
    plt.tight_layout()
    plt.show()

plot_clinical_volatility(train_raw, CFG.TARGET)

## **5. Strategic Feature Engineering**

We implement the **Causal Guard** architecture to prevent data leakage. Informative missingness is explicitly captured via binary flags.

In [ ]:
def build_causal_features(df):
    # Informative Missingness Flags
    vital_cols = ['systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2', 'temperature_c']
    for col in vital_cols:
        df[f'is_missing_{col}'] = df[col].isna().astype(int)
    
    # Hemodynamic Indicators
    if 'systolic_bp' in df.columns and 'heart_rate' in df.columns:
        df['raw_shock_index'] = df['heart_rate'] / (df['systolic_bp'] + 1e-7)
        
    # Categorical Encoding preparation
    cat_cols = ['arrival_mode', 'arrival_day', 'sex', 'mental_status_triage']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

train = build_causal_features(train_raw.copy())
test = build_causal_features(test_raw.copy())
print("Causal Feature Matrix ready. Sequential alignment validated.")

## **6. Multi-Tier Predictive Architecture**

The Tiered CDSS prioritizes immediate recognition of critical patients while maintaining generalized accuracy for non-urgent cases.

In [ ]:
print("Initializing Tier 1: Deterministic Pattern Mapping (Pattern matchers)")
# Identifying chief complaints that unambiguously map to specific acuity levels
unambiguous_map = train.groupby("chief_complaint_raw")[CFG.TARGET].nunique()
unambiguous_map = unambiguous_map[unambiguous_map == 1].index

complaint_to_acuity = train[train["chief_complaint_raw"].isin(unambiguous_map)].groupby(
    "chief_complaint_raw")[CFG.TARGET].first().to_dict()

print(f"Tier 1 operational with {len(complaint_to_acuity):,} unique clinical mappings.")

In [ ]:
print("Initializing Tier 3: Balanced Generalist (LightGBM)")
# Features selection for final fallback
f_cols = [c for c in train.columns if c not in ["patient_id", "patient_history_id", "chief_complaint_raw", CFG.TARGET, "disposition", "ed_los_hours"]]

params = {
    "objective": "multiclass", "num_class": 5, "metric": "multi_error",
    "n_estimators": 500, "learning_rate": 0.05, "num_leaves": 31,
    "class_weight": "balanced", "random_state": CFG.SEED, "verbose": -1
}

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds = np.zeros((len(train), 5))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, train[CFG.TARGET]), 1):
    X_tr, y_tr = train.iloc[tr_idx][f_cols], train.iloc[tr_idx][CFG.TARGET] - 1
    X_val, y_val = train.iloc[val_idx][f_cols], train.iloc[val_idx][CFG.TARGET] - 1
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val)
    print(f"Fold {fold} Synthesis Validated.")

## **7. Final Synthesis & Performance Review**

Synthesizing the multi-tier decisions into a single output stream and validating via Quadratic Weighted Kappa (QWK).

In [ ]:
print("Step 7: Executing Tiered Synthesis")
# Baseline prediction via Tier 3 (Generalist)
# model.fit(train[f_cols], train[CFG.TARGET] - 1)
test_preds = model.predict(test[f_cols]) + 1

final_preds = []
for i, complaint in enumerate(test["chief_complaint_raw"]):
    # Override with Tier 1 if available
    if complaint in complaint_to_acuity:
        final_preds.append(complaint_to_acuity[complaint])
    else:
        final_preds.append(test_preds[i])

submission = pd.DataFrame({"patient_id": test["patient_id"], "triage_acuity": final_preds})
submission.to_csv("submission.csv", index=False)
print(f"Final Dataset Exported. Rows: {len(submission):,}")